# Issue #15 [Data/Integration]: Integración Multifuente, Control de Duplicación en Joins, Reestructuración Tabular y Enlace con Datos Abiertos

**Subentorno:** `analytics/` | **Rama:** `feat/JMCM-analytics-nosql` | **Script equivalente:** `scripts/15_integracion_reestructuracion_yeb.py`  
**Criterios de Aceptación:** `CA-ISS-15-01` a `CA-ISS-15-06` | **Riesgos mitigados:** `NEG-012`, `TEC-019` | **Casos de prueba:** `CP-055`, `CP-055B`, `CP-055C`

Este cuaderno integra la base limpia de postulaciones aptas de las Semanas 5–7 (`n = 93`) con el catálogo maestro de vacantes/empresas (`many_to_one`), el log 1:N de eventos de seguimiento (pre-agregado con `groupby().agg()`), una segunda cohorte semestral longitudinal (`2026-B` vía `concat()`), la reestructuración bidireccional reversible (`melt()` ↔ `pivot()`) y el enlace contextual con Datos Abiertos de Colombia (`datos.gov.co`, SENA `28vu-5tx7`).

## 1. Línea base y preservación del universo analizable (`CA-ISS-15-01`)

Se recupera la base transformada `data/processed/yeb_postulaciones_transformado.csv` en modo de **solo lectura** (registrando su huella `SHA-256`), filtrando el universo analizable `n = 93` (`origen_registro == 'dataset_preparado'`) y cuadrando los KPIs exactos contra `outputs/yeb_kpis_semana6.csv`.

In [1]:
from pathlib import Path
import importlib, sys
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
if (cwd / 'data' / 'processed').exists():
    ANALYTICS_ROOT = cwd
elif (cwd / 'analytics' / 'data' / 'processed').exists():
    ANALYTICS_ROOT = cwd / 'analytics'
else:
    ANALYTICS_ROOT = cwd.parent

sys.path.insert(0, str(ANALYTICS_ROOT / 'scripts'))
m15 = importlib.import_module('15_integracion_reestructuracion_yeb')

df_total, df_u, ruta_fuente, huella_antes = m15.cargar_linea_base_semana7(ANALYTICS_ROOT)
kpi_base = m15.calcular_firma_kpis_referencia(df_u)

df_kpi_s6 = pd.read_csv(ANALYTICS_ROOT / 'outputs' / 'yeb_kpis_semana6.csv')
global_s6 = df_kpi_s6.loc[df_kpi_s6['dimension'] == 'Global'].iloc[0]
assert len(df_u) == int(global_s6['n_filas_size']) == 93
assert kpi_base['tasa_entrevista_pct'] == round(float(global_s6['tasa_entrevista_pct']), 2) == 39.78
assert kpi_base['tiempo_resp_media'] == round(float(global_s6['tiempo_resp_media']), 2) == 4.0

print(f'Fuente base: {ruta_fuente.name} | SHA-256: {huella_antes[:16]}...')
print(f'Universo analizable: n = {len(df_u)} | Candidatos únicos: {kpi_base["n_candidatos"]} | Vacantes: {kpi_base["n_vacantes"]}')
print(f'KPI-YEB-01 (entrevista/aceptada): {kpi_base["tasa_entrevista_pct"]}% | KPI-YEB-02 (resp. media): {kpi_base["tiempo_resp_media"]} días (count={kpi_base["n_respondidas"]})')

Fuente base: yeb_postulaciones_transformado.csv | SHA-256: 769e94c0a85a651f...
Universo analizable: n = 93 | Candidatos únicos: 93 | Vacantes: 18
KPI-YEB-01 (entrevista/aceptada): 39.78% | KPI-YEB-02 (resp. media): 4.0 días (count=55)


## 2. Integración con el catálogo maestro de vacantes y empresas (`merge()` auditado) (`CA-ISS-15-02`)

- Cruce `how='left'` con validación estricta de cardinalidad `validate='many_to_one'`.
- Prueba defensiva negativa: se inyecta un `job_id` duplicado en una copia del catálogo para confirmar que Pandas lanza `MergeError` y bloquea productos cartesianos.
- Auditoría con `indicator=True` (en `left` y `outer`) para identificar vacantes sin postulantes activos.

In [2]:
catalogo, ruta_cat = m15.generar_o_cargar_catalogo_vacantes(ANALYTICS_ROOT)
res_cat = m15.integrar_con_catalogo(df_u, catalogo)
df_con_cat = res_cat['df_con_catalogo']

assert len(df_con_cat) == 93
assert res_cat['merge_error_capturado'] is True
assert res_cat['conteo_merge_left'] == {'both': 93, 'left_only': 0, 'right_only': 0}

print('Catálogo cargado:', ruta_cat.name, '| Vacantes totales:', len(catalogo))
print('Prueba negativa (duplicado en catálogo -> MergeError):', res_cat['merge_error_capturado'])
print('Auditoría indicator=True (left join):', res_cat['conteo_merge_left'])
print('\nVacantes sin postulantes activos detectadas en auditoría outer (right_only):')
display(res_cat['vacantes_sin_postulantes'])

Catálogo cargado: yeb_catalogo_vacantes_empresas.csv | Vacantes totales: 21
Prueba negativa (duplicado en catálogo -> MergeError): True
Auditoría indicator=True (left join): {'both': 93, 'left_only': 0, 'right_only': 0}

Vacantes sin postulantes activos detectadas en auditoría outer (right_only):


,job_id,company_name,ciudad_sede,estado_vacante_catalogo
0,JOB-2026-019,Verde Digital Antioquia,Medellín,published
1,JOB-2026-020,ConectaJoven Digital S.A.S.,Bogotá,published
2,JOB-2026-021,Valle Conecta BPO,Cali,draft


## 3. Prevención de multiplicación de KPI ante eventos 1:N (`groupby().agg()`) (`CA-ISS-15-03`)

El log `yeb_eventos_seguimiento.csv` contiene `426` eventos de chat, revisiones de CV y notificaciones asociados a `88` de las `93` postulaciones. Primero se cuantifica la distorsión producida por un **join 1:N ingenuo** (antipatrón) y luego se aplica el patrón seguro: **agregar primero por `application_id`** antes del `merge(how='left', validate='one_to_one')`.

In [3]:
eventos, ruta_ev = m15.generar_o_cargar_eventos_seguimiento(ANALYTICS_ROOT, df_u)
res_ev = m15.integrar_eventos_sin_multiplicacion(df_con_cat, eventos, kpi_base)
df_integrado = res_ev['df_integrado']

assert len(df_integrado) == 93
assert res_ev['kpi_integrado'] == kpi_base

comparativa_1n = pd.DataFrame([
    {'escenario': 'Línea base Semana 6–7', 'filas': kpi_base['n_filas'], 'tasa_entrevista_pct': kpi_base['tasa_entrevista_pct'], 'estado': 'Referencia'},
    {'escenario': 'Antipatrón: join 1:N directo sin agregar', 'filas': res_ev['filas_join_ingenuo'], 'tasa_entrevista_pct': res_ev['tasa_entrevista_ingenua_pct'], 'estado': 'DISTORSIONADO (+363% filas)'},
    {'escenario': 'Correcto: groupby(application_id).agg + merge 1:1', 'filas': len(df_integrado), 'tasa_entrevista_pct': res_ev['kpi_integrado']['tasa_entrevista_pct'], 'estado': 'EXACTO (assert OK)'},
])
display(comparativa_1n)
print('Auditoría indicator=True eventos:', res_ev['conteo_merge_eventos'])
display(df_integrado[['application_id', 'status', 'company_name', 'total_eventos', 'n_mensajes_chat', 'n_revisiones_cv', 'latencia_media_eventos_min']].head(8))

,escenario,filas,tasa_entrevista_pct,estado
0,Línea base Semana 6–7,93,39.78,Referencia
1,Antipatrón: join 1:N directo sin agregar,431,56.38,DISTORSIONADO (+363% filas)
2,Correcto: groupby(application_id).agg + merge 1:1,93,39.78,EXACTO (assert OK)


Auditoría indicator=True eventos: {'both': 88, 'left_only': 5, 'right_only': 0}


,application_id,status,company_name,total_eventos,n_mensajes_chat,n_revisiones_cv,latencia_media_eventos_min
0,APP-2026-0001,under_review,ConectaJoven Digital S.A.S.,5,3,1,235.70
1,APP-2026-0002,interview,Pacífico Software Studio,5,3,1,257.28
2,APP-2026-0003,accepted,Logística Caribe Express,6,2,1,318.57
3,APP-2026-0004,rejected,Santander Innovación Digital,4,1,1,151.52
4,APP-2026-0005,submitted,Eje Cafetero Tech Labs,3,1,1,162.17
5,APP-2026-0006,under_review,Verde Digital Antioquia,4,2,1,292.32
6,APP-2026-0007,interview,Grupo Finanzas Andinas,5,3,1,161.98
7,APP-2026-0008,accepted,Pacífico Software Studio,8,4,1,317.44


## 4. Concatenación longitudinal de cohortes semestrales (`concat()`) (`CA-ISS-15-04`)

Se apilan las cohortes `2026-A` (`n = 93`) y `2026-B` (`n = 60`) tras verificar esquemas idénticos, comprobando con `assert` **`0` duplicados** sobre la llave compuesta `(candidate_hash, job_id, periodo)`.

In [4]:
cohorte_26b, ruta_26b = m15.generar_o_cargar_cohorte_2026b(ANALYTICS_ROOT, df_u)
res_long = m15.concatenar_cohortes_longitudinales(df_u, cohorte_26b)

assert len(res_long['df_longitudinal']) == 153
assert res_long['duplicados_llave_compuesta'] == 0

print(f'Total apilado longitudinal: {len(res_long["df_longitudinal"])} filas (93 + 60)')
print(f'Duplicados en llave compuesta (candidate_hash + job_id + periodo): {res_long["duplicados_llave_compuesta"]}')
print(f'Candidatos recurrentes entre 2026-A y 2026-B: {res_long["candidatos_recurrentes"]}')
display(res_long['resumen_periodos'])

Total apilado longitudinal: 153 filas (93 + 60)
Duplicados en llave compuesta (candidate_hash + job_id + periodo): 0
Candidatos recurrentes entre 2026-A y 2026-B: 12


,periodo,n_postulaciones,n_candidatos_unicos,tasa_entrevista_pct,salario_mediano_cop
0,2026-A,93,93,39.78,1972608.0
1,2026-B,60,60,41.67,2037308.5


## 5. Reestructuración bidireccional ancho ↔ largo (`melt()` y `pivot()`) (`CA-ISS-15-05`)

Se transforman los puntajes de evaluación (`score_habilidades`, `score_experiencia`, `score_disponibilidad`) desde formato ancho `(93, 6)` a formato largo `(279, 5)` mediante `melt()`, y se reconstruye la vista ancha original mediante `pivot()` demostrando reversibilidad matemática exacta (`max_diff == 0.0` y `assert_frame_equal`).

In [5]:
res_melt = m15.reestructurar_melt_y_pivot(df_integrado)

assert res_melt['df_ancho'].shape == (93, 6)
assert res_melt['df_largo'].shape == (279, 5)
assert res_melt['df_reconstruido'].shape == (93, 6)
assert res_melt['max_diff_absoluta'] == 0.0

print('Forma ancho original    :', res_melt['df_ancho'].shape)
print('Forma largo (melt)      :', res_melt['df_largo'].shape)
print('Forma reconstruida pivot:', res_melt['df_reconstruido'].shape)
print('Diferencia absoluta máx.:', res_melt['max_diff_absoluta'])
display(res_melt['df_largo'].head(6))
display(res_melt['resumen_componentes'])

Forma ancho original    : (93, 6)
Forma largo (melt)      : (279, 5)
Forma reconstruida pivot: (93, 6)
Diferencia absoluta máx.: 0.0


,application_id,candidate_hash,job_id,componente,puntaje
0,APP-2026-0001,CAND-HASH-0001,JOB-2026-002,score_habilidades,77.00
1,APP-2026-0002,CAND-HASH-0002,JOB-2026-003,score_habilidades,85.35
2,APP-2026-0003,CAND-HASH-0003,JOB-2026-004,score_habilidades,76.35
3,APP-2026-0004,CAND-HASH-0004,JOB-2026-005,score_habilidades,85.35
4,APP-2026-0005,CAND-HASH-0005,JOB-2026-006,score_habilidades,67.70
5,APP-2026-0006,CAND-HASH-0006,JOB-2026-007,score_habilidades,68.20


,componente,n,media,mediana,std,minimo,maximo
0,score_disponibilidad,93,84.89,85.5,4.27,73.21,93.56
1,score_experiencia,93,83.24,86.0,10.72,62.00,100.00
2,score_habilidades,93,80.75,81.3,6.85,67.70,93.65


## 6. Enlace contextual con Datos Abiertos de Colombia (`datos.gov.co` — SENA `28vu-5tx7`) (`CA-ISS-15-06`)

Se consulta el conjunto público oficial del SENA *«CERTIFICACIÓN DE LA FORMACIÓN PROFESIONAL INTEGRAL»* (`28vu-5tx7`, licencia `CC BY-SA 4.0`) mediante la API Socrata de `datos.gov.co`, manteniendo una tabla territorial paralela a grano departamento **sin cruzar ni sumar cifras macro agregadas sobre las postulaciones individuales del portal**.

In [6]:
res_sena = m15.consultar_datos_abiertos_sena(ANALYTICS_ROOT, df_u)
res_full = m15.ejecutar_flujo_completo(ANALYTICS_ROOT, verify_docs=True)

assert res_full['huella_fuente_sha256'] == huella_antes, 'La fuente base S5–S7 debe permanecer inmutable'

print(f'Origen consulta datos.gov.co : {res_sena["origen_consulta"]} (HTTP {res_sena["http_status"]})')
print(f'Total nacional SENA (202402) : {res_sena["total_nac_aprendices"]:,} aprendices en {res_sena["total_nac_fichas"]:,} fichas ({res_sena["total_nac_jovenes"]:,} jóvenes focalizados)')
print(f'Concentración 6 deptos YEB   : {res_sena["participacion_6_deps_pct"]}% de aprendices certificados nacionales')
display(res_sena['tabla_contexto'])

print('\n' + m15.CONCLUSION_DATOS_ABIERTOS)
print('\nBitácora consolidada de auditoría de integración:')
display(pd.read_csv(ANALYTICS_ROOT / 'outputs' / 'auditoria_integracion_joins_semana8.csv'))

Origen consulta datos.gov.co : api_live_datos_gov_co (HTTP 200)
Total nacional SENA (202402) : 61,420 aprendices en 8,538 fichas (7,014 jóvenes focalizados)
Concentración 6 deptos YEB   : 50.79% de aprendices certificados nacionales


,departamento,yeb_postulaciones_2026A,yeb_tasa_entrevista_pct,yeb_salario_mediano_cop,departamento_sena,yeb_participacion_pct,fichas_sena,total_aprendices_sena,total_jovenes_focalizados_sena,sena_participacion_nacional_pct,nota_metodologica
0,Bogotá D.C.,17,41.18,1972608.0,BOGOTA D.C.,18.28,1385,9924,662,16.16,Lectura territorial paralela: microdatos YEB (...
1,Antioquia,15,40.00,1843002.0,ANTIOQUIA,16.13,1256,9911,830,16.14,Lectura territorial paralela: microdatos YEB (...
2,Valle del Cauca,16,50.00,2023486.0,VALLE DEL CAUCA,17.20,636,5447,687,8.87,Lectura territorial paralela: microdatos YEB (...
3,Atlántico,15,40.00,2088823.0,ATLANTICO,16.13,616,2170,606,3.53,Lectura territorial paralela: microdatos YEB (...
4,Santander,14,28.57,2233809.5,SANTANDER,15.05,388,2110,90,3.44,Lectura territorial paralela: microdatos YEB (...
5,Risaralda,16,37.50,1843937.5,RISARALDA,17.20,187,1633,347,2.66,Lectura territorial paralela: microdatos YEB (...



Conclusión técnica sobre el enlace contextual con Datos Abiertos (SENA 28vu-5tx7, datos.gov.co): El conjunto público de Certificación de la Formación Profesional Integral del SENA (periodo 202402, 8.538 fichas agregadas y 61.420 aprendices certificados a nivel nacional, de los cuales 7.014 pertenecen a la categoría específica de población joven focalizada) confirma que los seis departamentos atendidos por YEB (Bogotá D.C., Antioquia, Valle del Cauca, Atlántico, Santander y Risaralda) concentran más de la mitad de la oferta certificada nacional (31.195 aprendices, 50,79%), con fuerte predominio de formación presencial (93,29%). Sin embargo, sus cifras macro agregadas por ficha NO deben cruzarse fila a fila ni sumarse con los registros individuales del portal YEB (n=93 postulaciones en cohorte 2026-A): difieren en unidad de observación (ficha agregada vs. postulación individual), ventana temporal (2024 vs. 2026), definición operativa de juventud (marca poblacional vulnerable del SENA vs

,etapa,operacion,filas_antes,filas_despues,validacion_cardinalidad,tasa_entrevista_pct,hallazgo
0,1_linea_base_semana7,Carga de solo lectura (origen_registro == data...,97,93,application_id único (93/93),39.78,Universo analizable n=93 verificado contra yeb...
1,2_merge_catalogo_vacantes,"merge(on='job_id', how='left', validate='many_...",93,93,many_to_one OK + MergeError probado en duplicado,39.78,"93 both, 0 left_only; auditoría outer detecta ..."
2,3a_antipatron_join_1N_ingenuo,merge directo sin pre-agregar eventos (solo di...,93,431,FALLA ESPERADA: explosión 1:N,56.38,Multiplicó filas de 93 a 431 y distorsionó tas...
3,3b_join_eventos_preagregados,groupby(['application_id']).agg(...) + merge(h...,93,93,one_to_one OK (88 both + 5 sin eventos),39.78,0 filas multiplicadas; KPIs de Semanas 6 y 7 1...
4,4_concat_longitudinal,"concat([cohorte_2026A, cohorte_2026B], ignore_...",93,153,"0 duplicados en (candidate_hash, job_id, periodo)",40.52,93 (2026-A) + 60 (2026-B) = 153 filas; 12 cand...
5,5_melt_y_pivot_reversible,melt() -> 279 filas largas -> pivot() -> 93 fi...,93,93,max_diff_absoluta = 0.0,39.78,Reversibilidad exacta verificada con assert_fr...
6,6_datos_abiertos_sena,"Consulta Socrata datos.gov.co (28vu-5tx7, api_...",93,6,"Tabla territorial paralela (6 departamentos, s...",39.78,SENA 202402: 61420 aprendices en 8538 fichas; ...
